<a href="https://colab.research.google.com/github/K-vino/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/K-vino/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My baseline rule

I will prioritize webpages for content-refresh review using three historical signals:

- `days_since_last_update`: older pages receive higher priority.
- `avg_position`: pages with weaker search positions receive higher priority.
- `ctr`: pages with lower CTR receive higher priority.

Scoring rule:
- +2 points if `days_since_last_update >= 365`
- +1 point if `avg_position > 20`
- +1 point if `ctr < 0.10`

Higher scores are ranked first.

Reason code: `refresh_review`

Action: `Review for content refresh`

This is a transparent baseline using historical information available at the decision moment. It is directional decision-support and does not guarantee that refreshing a page will improve performance.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
%cd /content

!git clone https://github.com/K-vino/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship

!ls data/raw

/content
Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 148 (delta 59), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 1.86 MiB | 16.02 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/flyrank-ml-internship
content_refresh_anonymized.csv


In [10]:
import pandas as pd
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Number of webpages:", len(df))
print("Signals used:")
print(["days_since_last_update", "avg_position", "ctr"])

Number of webpages: 30000
Signals used:
['days_since_last_update', 'avg_position', 'ctr']


In [13]:
# Signal check 1: content age
age_bucket = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, float("inf")],
    labels=["0-90", "91-180", "181-365", "365+"]
)

print("Signal 1: days_since_last_update")
print(
    age_bucket.value_counts(dropna=False)
    .sort_index()
    .rename_axis("age_bucket")
    .reset_index(name="n")
)

Signal 1: days_since_last_update
  age_bucket      n
0       0-90  20655
1     91-180   9171
2    181-365    169
3       365+      5


In [14]:
# Signal check 2: search position
position_bucket = pd.cut(
    df["avg_position"],
    bins=[0, 10, 20, 50, float("inf")],
    labels=["1-10", "10-20", "20-50", "50+"]
)

print("Signal 2: avg_position")
print(
    position_bucket.value_counts(dropna=False)
    .sort_index()
    .rename_axis("position_bucket")
    .reset_index(name="n")
)

Signal 2: avg_position
  position_bucket      n
0            1-10  12983
1           10-20   7273
2           20-50   7225
3             50+   1314
4             NaN   1205


Signal 1 verdict: CONFIRMED

Older pages are present in the dataset, including pages 365+ days since update.
This supports using content age as a directional refresh-priority signal.

Signal 2 verdict: CONFIRMED

The dataset contains pages across different average-position buckets.
This supports using search position as a directional prioritization signal.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
df["baseline_score"] = (
    (df["days_since_last_update"] >= 365).astype(int) * 2
    + (df["avg_position"] > 20).astype(int)
    + (df["ctr"] < 0.10).astype(int)
)

df["reason_code"] = "refresh_review"
df["action"] = "Review for content refresh"

queue = df.sort_values(
    ["baseline_score", "days_since_last_update", "avg_position"],
    ascending=[False, False, False]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

baseline_queue = queue[
    ["rank", "content_id", "baseline_score", "reason_code", "action"]
]

os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows ranked:", len(baseline_queue))
print("CSV created:", os.path.exists(
    "work/outputs/baseline_action_score.csv"
))

display(baseline_queue.head(20))

Rows ranked: 30000
CSV created: True


,rank,content_id,baseline_score,reason_code,action
0,1,content_f6fdf87348f6,4,refresh_review,Review for content refresh
1,2,content_8d56efff1e71,4,refresh_review,Review for content refresh
2,3,content_55a5b1c46474,3,refresh_review,Review for content refresh
3,4,content_1b4ec72dafd4,3,refresh_review,Review for content refresh
4,5,content_3f3576c295f5,2,refresh_review,Review for content refresh
5,6,content_6476d1d8c050,2,refresh_review,Review for content refresh
6,7,content_7a888d3d99c8,2,refresh_review,Review for content refresh
7,8,content_d25a099b3726,2,refresh_review,Review for content refresh
8,9,content_dd413158df3c,2,refresh_review,Review for content refresh
9,10,content_afd26a07382d,2,refresh_review,Review for content refresh


### Top-20 review

1. Rank 1 — Review for content refresh | refresh_review | Moderate confidence. Old page, weak position, and very low CTR. Could be wrong if the page is intentionally evergreen or already scheduled for update.

2. Rank 2 — Review for content refresh | refresh_review | Moderate confidence. Old page with weak position and very low CTR. Could be wrong if search demand has changed for reasons unrelated to content.

3. Rank 3 — Review for content refresh | refresh_review | Moderate confidence. Old page and very low CTR. Could be wrong if the low CTR is caused by the search result layout rather than page quality.

4. Rank 4 — Review for content refresh | refresh_review | Moderate confidence. Old page and very low CTR. Could be wrong if the page serves a low-click informational query.

5. Rank 5 — Review for content refresh | refresh_review | Low-to-moderate confidence. The page is old, but its position is strong. Could be a weak pick because the page already ranks well.

6. Rank 6 — Review for content refresh | refresh_review | Moderate confidence. Old enough to review and weak position. Could be wrong if the page has recently been reviewed outside this dataset.

7. Rank 7 — Review for content refresh | refresh_review | Moderate confidence. Old page with weak position and low CTR. Could be wrong if external ranking factors explain the performance.

8. Rank 8 — Review for content refresh | refresh_review | Moderate confidence. Old page with weak position and low CTR. Could be wrong if the query intent has changed.

9. Rank 9 — Review for content refresh | refresh_review | Moderate confidence. Old page with weak position and low CTR. Could be wrong if the page is intentionally targeting a narrow audience.

10. Rank 10 — Review for content refresh | refresh_review | Moderate confidence. Old page with weak position and low CTR. Could be wrong if the traffic opportunity is too small.

11. Rank 11 — Review for content refresh | refresh_review | Moderate confidence. Old page and weak position. Could be wrong if the page is not strategically important.

12. Rank 12 — Review for content refresh | refresh_review | Moderate confidence. Old page with weak position and low CTR. Could be wrong if the ranking is caused by factors outside the content.

13. Rank 13 — Review for content refresh | refresh_review | Moderate confidence. Old page with weak position and low CTR. Could be wrong if the page has limited search demand.

14. Rank 14 — Review for content refresh | refresh_review | Moderate confidence. Old page with weak position and low CTR. Could be wrong if the page is already being maintained.

15. Rank 15 — Review for content refresh | refresh_review | Moderate confidence. Old page with weak position and low CTR. Could be wrong if low CTR reflects query intent rather than content quality.

16. Rank 16 — Review for content refresh | refresh_review | Moderate confidence. Old page with weak position and low CTR. Could be wrong if another business priority should come first.

17. Rank 17 — Review for content refresh | refresh_review | Moderate confidence. Low CTR and weak position increase review priority. Could be wrong if the page has little practical refresh opportunity.

18. Rank 18 — Review for content refresh | refresh_review | Moderate confidence. Low CTR and weak position suggest review. Could be wrong if the page's search demand is declining.

19. Rank 19 — Review for content refresh | refresh_review | Moderate confidence. Weak position and low CTR suggest review. Could be wrong if the page is intentionally low-volume.

20. Rank 20 — Review for content refresh | refresh_review | Moderate confidence. Weak position and low CTR suggest review. Could be wrong if the page has business context not represented in the dataset.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:
top20 = queue.head(20)[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "avg_position",
        "ctr"
    ]
]

display(top20)

,rank,content_id,baseline_score,reason_code,action,days_since_last_update,avg_position,ctr
0,1,content_f6fdf87348f6,4,refresh_review,Review for content refresh,373,32.5,0.00
1,2,content_8d56efff1e71,4,refresh_review,Review for content refresh,372,35.0,0.00
2,3,content_55a5b1c46474,3,refresh_review,Review for content refresh,373,7.5,0.00
3,4,content_1b4ec72dafd4,3,refresh_review,Review for content refresh,372,7.0,0.00
4,5,content_3f3576c295f5,2,refresh_review,Review for content refresh,373,1.0,100.00
5,6,content_6476d1d8c050,2,refresh_review,Review for content refresh,313,67.8,0.00
6,7,content_7a888d3d99c8,2,refresh_review,Review for content refresh,313,67.6,0.00
7,8,content_d25a099b3726,2,refresh_review,Review for content refresh,305,64.5,0.00
8,9,content_dd413158df3c,2,refresh_review,Review for content refresh,305,46.1,0.00
9,10,content_afd26a07382d,2,refresh_review,Review for content refresh,305,45.1,0.00


In [15]:
# Leakage check
forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

used_columns = [
    "days_since_last_update",
    "avg_position",
    "ctr"
]

print("Used columns:", used_columns)

leaked = [c for c in forbidden_columns if c in used_columns]

print("Forbidden label-derived columns used:", leaked)

assert len(leaked) == 0, "Potential leakage detected!"

print("Leakage check: PASSED")

Used columns: ['days_since_last_update', 'avg_position', 'ctr']
Forbidden label-derived columns used: []
Leakage check: PASSED


### Leakage check

The baseline uses only historical signals:
days_since_last_update, avg_position, and ctr.

No future-window outcome or label-derived field was used in the score.
The baseline is therefore intended as a pre-decision rule, not a future-outcome prediction.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [16]:
import os

print(os.path.exists("work/notebooks/w04_baseline_score.ipynb"))
print(os.path.exists("work/outputs/baseline_action_score.csv"))

True
True


## Top-20 review

Each top-ranked webpage is a review candidate, not an automatic update.

Confidence is moderate because the baseline uses only three historical signals.

A recommendation could be wrong if:
- the page is already scheduled for an update,
- low CTR has another cause,
- search position is affected by factors outside the content,
- or the page has business context not represented in the dataset.

For each of the 20 rows above, I will review the action, reason code, confidence, and possible failure reason.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.